## **Machine Learning Aplicado a las Finanzas** 🚀
### **HW Sesión 3**

Andrés C. Medina Sanhueza

Senior Data Scientist Engineer

anmedinas@gmail.com

---

### Consideraciones Previas

* Tarea es **Individual**
* Fecha de Entrega: **Jueves 21 May 23:59** (cualquier commit posterior descontará puntos)
* El notebook debe correr de arriba abajo sin errores (`Kernel → Restart & Run All`)
* No se admiten `pass` pendientes ni celdas vacías donde se espera código
* **Regla de oro:** si se detecta data leakage en alguna solución, esa parte se evalúa con nota mínima
* Deben crear una rama nueva en su repositorio `feat/hw03` y deben trabajar la tarea en esta rama. Deben dejar el notebook en la carpeta `notebooks`
* **Archivo a entregar:** URL del repositorio GitHub enviada a `anmedinas@gmail.com`  

---
## Parte 1 — Auditoría de Data Leakage *(25 pts)*

### Contexto

El siguiente bloque contiene **tres formas de data leakage** deliberadamente introducidas en un pipeline de predicción de dirección diaria del SPY. Tu tarea es identificarlas, corregirlas y cuantificar el impacto en el AUC.

### 1.1 — Identifica los leakages *(8 pts)*

Ejecuta el pipeline defectuoso y completa la celda de análisis indicando:
- El tipo de leakage (normalización, rolling, label)
- La línea exacta del código que lo produce
- Por qué es un problema en un contexto de trading real

In [ ]:
import numpy as np
import pandas as pd
import yfinance as yf
import warnings
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.model_selection import TimeSeriesSplit, GridSearchCV
from sklearn.metrics import roc_auc_score
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.base import clone
from xgboost import XGBClassifier

warnings.filterwarnings('ignore')
sns.set_style('dark')
plt.rcParams.update({'figure.dpi': 120, 'axes.spines.top': False, 'axes.spines.right': False})

raw = yf.download('SPY', start='2015-01-01', end='2024-12-31', auto_adjust=True, progress=False)
ret = raw['Close'].squeeze().pct_change().dropna()
ret.name = 'ret'
print(f'Serie: {ret.index[0].date()} — {ret.index[-1].date()} | {len(ret):,} observaciones')

In [ ]:
def build_pipeline_bad(ret):
    """
    Pipeline defectuoso con tres leakages ocultos.
    NO uses esta funcion en produccion.
    """
    df = pd.DataFrame({'ret': ret})

    # LEAKAGE A: rolling centrado (incluye datos futuros t+1, t+2, ...)
    df['vol_20']   = df['ret'].rolling(20, center=True).std()
    df['mean_10']  = df['ret'].rolling(10, center=True).mean()
    df['ret_lag1'] = df['ret'].shift(1)

    # LEAKAGE B: label sin desplazamiento — predice el retorno de HOY usando features de HOY
    df['label'] = (df['ret'] > 0).astype(int)

    df = df.dropna()
    X  = df[['ret_lag1', 'vol_20', 'mean_10']].values
    y  = df['label'].values

    # LEAKAGE C: StandardScaler ajustado sobre toda la serie antes del split
    scaler   = StandardScaler()
    X_scaled = scaler.fit_transform(X)

    cut = int(len(X_scaled) * 0.7)
    return X_scaled[:cut], X_scaled[cut:], y[:cut], y[cut:]


X_tr_bad, X_te_bad, y_tr_bad, y_te_bad = build_pipeline_bad(ret)
model_bad = LogisticRegression(C=0.1, max_iter=1000).fit(X_tr_bad, y_tr_bad)
auc_bad   = roc_auc_score(y_te_bad, model_bad.predict_proba(X_te_bad)[:, 1])
print(f'AUC pipeline defectuoso: {auc_bad:.4f}')

### 1.2 — Análisis de leakages identificados *(8 pts)*

Completa la siguiente tabla con tus respuestas:

| # | Tipo de leakage | Línea del código | Problema en trading real |
|---|-----------------|------------------|--------------------------|
| A | Rolling leakage | `df['vol_20'] = df['ret'].rolling(20, center=True).std()` y `df['mean_10'] = df['ret'].rolling(10, center=True).mean()` | Al usar `center=True`, cada feature de la fecha `t` incorpora retornos futuros (`t+1`, `t+2`, etc.). En trading real esos datos aun no existen al momento de decidir, por lo que el AUC defectuoso de 0.6214 probablemente esta inflado. |
| B | Label leakage | `df['label'] = (df['ret'] > 0).astype(int)` | El target usa el retorno del mismo dia que participa indirectamente en las features centradas. Para una senal operable se debe predecir la direccion futura, por ejemplo `ret_{t+1}`; de lo contrario el modelo aprende una relacion contemporanea que no podria explotarse al cierre real de la decision. |
| C | Normalizacion leakage | `X_scaled = scaler.fit_transform(X)` antes del split temporal | El `StandardScaler` calcula media y desviacion usando train y test completos. Eso transfiere informacion de la distribucion futura al entrenamiento y contribuye a que el resultado reportado (`AUC = 0.6214`) no represente un desempeno out-of-time honesto. |

**Impacto observado:** al corregir los tres leakages, el AUC baja desde `0.6214` hasta `0.4782`. La diferencia de `+0.1433` muestra que el pipeline defectuoso sobreestima de forma importante la capacidad predictiva del modelo.

### 1.3 — Implementa el pipeline correcto *(9 pts)*

Implementa `build_pipeline_good()` corrigiendo los tres leakages. Usa `sklearn.Pipeline` para garantizar que el `StandardScaler` solo se ajusta en train. Imprime la diferencia de AUC entre el pipeline defectuoso y el correcto.

In [ ]:
def build_pipeline_good(ret):
    """
    Pipeline corregido sin data leakage.
    Retorna (pipe_fitted, X_train_raw, X_test_raw, y_train, y_test)
    """
    r = ret.squeeze() if isinstance(ret, pd.DataFrame) else ret
    df = pd.DataFrame({'ret': r})

    r_lagged = df['ret'].shift(1)
    df['vol_20']   = r_lagged.rolling(20, center=False).std()
    df['mean_10']  = r_lagged.rolling(10, center=False).mean()
    df['ret_lag1'] = r_lagged
    df['label']    = (df['ret'].shift(-1) > 0).astype(int)

    df = df.dropna()
    X  = df[['ret_lag1', 'vol_20', 'mean_10']]
    y  = df['label']

    cut = int(len(X) * 0.7)
    X_train, X_test = X.iloc[:cut], X.iloc[cut:]
    y_train, y_test = y.iloc[:cut], y.iloc[cut:]

    pipe = Pipeline([
        ('scaler', StandardScaler()),
        ('model', LogisticRegression(C=0.1, max_iter=1000)),
    ])
    pipe.fit(X_train, y_train)

    return pipe, X_train, X_test, y_train, y_test


pipe_good, X_tr_g, X_te_g, y_tr_g, y_te_g = build_pipeline_good(ret)
auc_good = roc_auc_score(y_te_g, pipe_good.predict_proba(X_te_g)[:, 1])
print(f'AUC correcto      : {auc_good:.4f}')
print(f'AUC defectuoso    : {auc_bad:.4f}')
print(f'Inflacion leakage : {auc_bad - auc_good:+.4f}')

---
## Parte 2 — Walk-Forward con Rolling Window *(30 pts)*

### Contexto

En clase implementamos Walk-Forward con **Expanding Window** (el train crece acumulando toda la historia). En esta parte implementarás la variante **Rolling Window**, que mantiene un tamaño fijo de train y descarta observaciones antiguas al avanzar. Usarás datos de **AAPL (2015–2024)**.

### 2.1 — Construcción de features *(5 pts)*

Completa `build_features_aapl()` con al menos **8 features** anti-leakage:
- Retornos rezagados: lag 1, 2, 5
- Volatilidad rolling: 10 y 20 días (con `shift(1)` correcto)
- RSI de 14 días
- Skewness rolling 20 días
- Label: dirección del retorno en `t+1`

In [ ]:
raw_aapl = yf.download('AAPL', start='2015-01-01', end='2024-12-31',
                        auto_adjust=True, progress=False)
ret_aapl = raw_aapl['Close'].squeeze().pct_change().dropna()
ret_aapl.name = 'ret'
print(f'AAPL: {ret_aapl.index[0].date()} — {ret_aapl.index[-1].date()} | {len(ret_aapl):,} obs')


def _rsi(ret, window=14):
    delta = ret
    gain  = delta.clip(lower=0).rolling(window).mean()
    loss  = (-delta.clip(upper=0)).rolling(window).mean()
    rs    = gain / (loss + 1e-9)
    return 100 - (100 / (1 + rs))


def build_features_aapl(ret, horizonte=1):
    """
    Features anti-leakage para AAPL.
    Todos los features deben usar solo informacion disponible en t.
    """
    r  = ret.squeeze().copy() if isinstance(ret, pd.DataFrame) else ret.copy()
    df = pd.DataFrame(index=r.index)

    r_lagged = r.shift(1)

    df['ret_lag1'] = r.shift(1)
    df['ret_lag2'] = r.shift(2)
    df['ret_lag5'] = r.shift(5)
    df['vol_10']   = r_lagged.rolling(10).std()
    df['vol_20']   = r_lagged.rolling(20).std()
    df['mean_10']  = r_lagged.rolling(10).mean()
    df['rsi_14']   = _rsi(r_lagged, window=14)
    df['skew_20']  = r_lagged.rolling(20).skew()
    df['label']    = (r.shift(-horizonte) > 0).astype(int)

    return df.dropna()


feat = build_features_aapl(ret_aapl)
X = feat.drop(columns=['label'])
y = feat['label']
print(f'Dataset: {X.shape[0]} obs | {X.shape[1]} features | {y.mean():.2%} dias alcistas')

### 2.2 — Implementa WalkForwardRolling *(15 pts)*

Implementa la clase `WalkForwardRolling`. A diferencia de `TimeSeriesSplit`, la ventana de train tiene siempre exactamente `train_size` observaciones (no crece).

**Requisitos:**
- `.split(X)` es un generador que retorna `(array_train_idx, array_test_idx)`
- `.get_n_splits(X)` retorna el número total de folds
- El gap entre `train[-1]` y `test[0]` debe ser exactamente `gap + 1` posiciones
- El test unitario al final debe pasar sin modificaciones

In [ ]:
class WalkForwardRolling:
    """
    Walk-Forward con ventana de entrenamiento de tamano fijo.

    Parameters
    ----------
    train_size : int  — tamano fijo de la ventana de train
    test_size  : int  — tamano del bloque de test por fold
    gap        : int  — observaciones descartadas entre fin de train y comienzo de test
    step       : int  — avance entre folds (default = test_size)
    """

    def __init__(self, train_size, test_size, gap=0, step=None):
        if train_size <= 0:
            raise ValueError('train_size debe ser positivo')
        if test_size <= 0:
            raise ValueError('test_size debe ser positivo')
        if gap < 0:
            raise ValueError('gap no puede ser negativo')

        self.train_size = int(train_size)
        self.test_size  = int(test_size)
        self.gap        = int(gap)
        self.step       = int(test_size if step is None else step)

        if self.step <= 0:
            raise ValueError('step debe ser positivo')

    def split(self, X):
        """
        Generador: devuelve (train_indices, test_indices) por fold.
        La ventana de train siempre tiene self.train_size observaciones.
        """
        n = len(X)
        train_start = 0

        while True:
            train_end  = train_start + self.train_size
            test_start = train_end + self.gap
            test_end   = test_start + self.test_size

            if test_end > n:
                break

            train_idx = np.arange(train_start, train_end)
            test_idx  = np.arange(test_start, test_end)
            yield train_idx, test_idx

            train_start += self.step

    def get_n_splits(self, X):
        """Numero total de folds."""
        return sum(1 for _ in self.split(X))


def _test_wfr():
    n = 300
    X_dummy = np.arange(n)
    wfr     = WalkForwardRolling(train_size=80, test_size=20, gap=5, step=10)
    splits  = list(wfr.split(X_dummy))
    assert len(splits) > 0, 'No se generaron folds'
    for tr, te in splits:
        assert len(tr) == 80, f'Train size incorrecto: {len(tr)} != 80'
        assert len(te) == 20, f'Test size incorrecto: {len(te)} != 20'
        assert te[0] - tr[-1] > 5, f'Gap no respetado: {te[0] - tr[-1]}'
    assert wfr.get_n_splits(X_dummy) == len(splits), 'get_n_splits inconsistente'
    print(f'Test WalkForwardRolling: OK — {len(splits)} folds generados')

_test_wfr()

### 2.3 — Expanding vs Rolling: comparación empírica *(10 pts)*

Usando el dataset de AAPL con los features de 2.1, ejecuta ambas estrategias y genera:
1. Evolución temporal del AUC fold a fold (líneas superpuestas)
2. Boxplot comparativo de la distribución de AUC
3. Tabla resumen con AUC media, std y porcentaje de folds con AUC > 0.5

Parámetros sugeridos: `initial_train=252`, `rolling_train=252`, `test_size=63`, `gap=5`, `step=21`

In [ ]:
INITIAL_TRAIN = 252
ROLLING_TRAIN = 252
TEST_SIZE     = 63
GAP           = 5
STEP          = 21

pipe_base = Pipeline([
    ('imp',    SimpleImputer()),
    ('scaler', StandardScaler()),
    ('model',  LogisticRegression(C=0.1, max_iter=1000)),
])

# Walk-Forward Expanding: el train acumula toda la historia disponible.
aucs_expanding  = []
dates_expanding = []

end = INITIAL_TRAIN
while end + GAP + TEST_SIZE <= len(X):
    X_tr = X.iloc[:end]
    y_tr = y.iloc[:end]
    X_te = X.iloc[end + GAP : end + GAP + TEST_SIZE]
    y_te = y.iloc[end + GAP : end + GAP + TEST_SIZE]

    pipe_i = clone(pipe_base)
    pipe_i.fit(X_tr, y_tr)
    auc_i = roc_auc_score(y_te, pipe_i.predict_proba(X_te)[:, 1])

    aucs_expanding.append(auc_i)
    dates_expanding.append(X_te.index[0])
    end += STEP

# Walk-Forward Rolling: el train mantiene tamano fijo y avanza en el tiempo.
aucs_rolling  = []
dates_rolling = []

wfr = WalkForwardRolling(
    train_size=ROLLING_TRAIN,
    test_size=TEST_SIZE,
    gap=GAP,
    step=STEP,
)

for tr_idx, te_idx in wfr.split(X):
    X_tr = X.iloc[tr_idx]
    y_tr = y.iloc[tr_idx]
    X_te = X.iloc[te_idx]
    y_te = y.iloc[te_idx]

    pipe_i = clone(pipe_base)
    pipe_i.fit(X_tr, y_tr)
    auc_i = roc_auc_score(y_te, pipe_i.predict_proba(X_te)[:, 1])

    aucs_rolling.append(auc_i)
    dates_rolling.append(X_te.index[0])

resumen = pd.DataFrame({
    'Estrategia': ['Expanding', 'Rolling'],
    'AUC media' : [np.mean(aucs_expanding), np.mean(aucs_rolling)],
    'AUC std'   : [np.std(aucs_expanding),  np.std(aucs_rolling)],
    'Folds>0.5' : [f'{np.mean([a > 0.5 for a in aucs_expanding]):.0%}',
                   f'{np.mean([a > 0.5 for a in aucs_rolling]):.0%}'],
    'N folds'   : [len(aucs_expanding), len(aucs_rolling)],
})
try:
    display(resumen.round(4))
except NameError:
    print(resumen.round(4).to_string(index=False))

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(dates_expanding, aucs_expanding, marker='o', ms=3,
             lw=1.5, label='Expanding', color='#1f77b4')
axes[0].plot(dates_rolling, aucs_rolling, marker='o', ms=3,
             lw=1.5, label='Rolling', color='#ff7f0e')
axes[0].axhline(0.5, ls='--', color='gray', lw=1, label='No-skill')
axes[0].set_title('AAPL - AUC fold a fold')
axes[0].set_xlabel('Inicio del test')
axes[0].set_ylabel('AUC ROC')
axes[0].legend()
axes[0].tick_params(axis='x', rotation=35)

box = axes[1].boxplot([aucs_expanding, aucs_rolling], labels=['Expanding', 'Rolling'],
                      patch_artist=True, showmeans=True)
for patch, color in zip(box['boxes'], ['#1f77b4', '#ff7f0e']):
    patch.set_facecolor(color)
    patch.set_alpha(0.55)
axes[1].axhline(0.5, ls='--', color='gray', lw=1)
axes[1].set_title('Distribucion de AUC por estrategia')
axes[1].set_ylabel('AUC ROC')

plt.tight_layout()
plt.show()

**Responde (mínimo 3 oraciones):** ¿Cuál estrategia es más apropiada para AAPL y por qué?

> *Completa aquí*

---
## Parte 3 — Nested CV Temporal: Comparación de Modelos *(25 pts)*

### Contexto

En clase se usó Nested CV con un único modelo (Regresión Logística) para tuning de hiperparámetros. Aquí extenderás ese framework para comparar una **familia completa de modelos** usando datos de **JPM (2015–2024)**:

| Modelo | Complejidad | Hiperparámetros a explorar |
|--------|-------------|---------------------------|
| `LogisticRegression` | Lineal (baseline) | `C` |
| `SVC` | Kernel / margen | `C`, `kernel` |
| `RandomForestClassifier` | Ensemble bagging | `n_estimators`, `max_depth` |
| `GradientBoostingClassifier` | Ensemble boosting | `n_estimators`, `max_depth` |
| `XGBClassifier` | Boosting optimizado | `n_estimators`, `max_depth`, `learning_rate` |

Incluirás features **cross-asset** (correlación rolling JPM–SPY, nivel del VIX) que capturan el contexto de mercado al momento de la decisión.

### 3.1 — Dataset multi-feature con cross-asset *(8 pts)*

In [ ]:
tickers   = ['JPM', 'SPY', '^VIX']
raw_multi = yf.download(tickers, start='2015-01-01', end='2024-12-31',
                        auto_adjust=True, progress=False)
close = raw_multi['Close'].dropna(how='all')
rets  = close.pct_change().dropna()
print(f'Multi-asset: {rets.index[0].date()} — {rets.index[-1].date()}')
rets.head(3)

In [ ]:
def build_features_jpm(rets, horizonte=1):
    """
    Features para JPM con variables propias y cross-asset.

    Minimo 10 features anti-leakage:
    - JPM    : lags 1, 2, 5; vol_10, vol_20; rsi_14
    - SPY    : ret_spy_lag1; ret_spy_rolling_mean_5
    - VIX    : vix_lag1
    - Relacion: correlacion_rolling_20_jpm_spy
    - Label  : direccion de JPM en t+horizonte
    """
    jpm = rets['JPM'].copy()
    spy = rets['SPY'].copy()
    vix = rets['^VIX'].copy()

    df = pd.DataFrame(index=jpm.index)

    # TODO: implementa los features
    # Correlacion rolling JPM-SPY:
    #   jpm.shift(1).rolling(20).corr(spy.shift(1))

    pass


# feat_jpm = build_features_jpm(rets)
# X_jpm = feat_jpm.drop(columns=['label'])
# y_jpm = feat_jpm['label']
# print(f'JPM: {X_jpm.shape[0]} obs | {X_jpm.shape[1]} features')

### 3.2 — Nested CV Temporal para selección de modelo *(12 pts)*

Implementa el Nested CV Walk-Forward con los cinco candidatos definidos en el diccionario `candidates`. En cada fold externo:
1. Para cada candidato: `GridSearchCV(cv=TimeSeriesSplit(3), scoring='roc_auc')` **solo sobre `X_train`** del fold
2. El modelo ganador del fold es el de mayor `best_score_` en el loop interno
3. Evalúa el ganador en `X_test` del fold externo → `auc_outer`
4. Registra: `fold`, `test_start`, `best_model`, `auc_inner`, `auc_outer`, `brecha = inner − outer`

**Restricción estricta:** el test del fold externo nunca puede participar en el tuning interno.

In [ ]:
INITIAL_TRAIN_NCV = 504
TEST_SIZE_NCV     = 63
STEP_NCV          = 21
GAP_NCV           = 5

# Cinco candidatos con sus grids de hiperparámetros
candidates = {
    'LogisticRegression': (
        Pipeline([('imp', SimpleImputer()), ('scaler', StandardScaler()),
                  ('model', LogisticRegression(max_iter=1000))]),
        {'model__C': [0.01, 0.1, 1.0]}
    ),
    'SVM': (
        Pipeline([('imp', SimpleImputer()), ('scaler', StandardScaler()),
                  ('model', SVC(probability=True, random_state=42))]),
        {'model__C': [0.1, 1.0, 10.0], 'model__kernel': ['rbf', 'linear']}
    ),
    'RandomForest': (
        Pipeline([('imp', SimpleImputer()),
                  ('model', RandomForestClassifier(random_state=42))]),
        {'model__n_estimators': [100, 200], 'model__max_depth': [3, 5, None]}
    ),
    'GradientBoosting': (
        Pipeline([('imp', SimpleImputer()),
                  ('model', GradientBoostingClassifier(random_state=42))]),
        {'model__n_estimators': [50, 100], 'model__max_depth': [2, 3]}
    ),
    'XGBoost': (
        Pipeline([('imp', SimpleImputer()),
                  ('model', XGBClassifier(random_state=42, eval_metric='logloss',
                                          verbosity=0, use_label_encoder=False))]),
        {'model__n_estimators': [50, 100], 'model__max_depth': [3, 5],
         'model__learning_rate': [0.05, 0.1]}
    ),
}

# TODO: implementa el Nested CV Walk-Forward
nested_results = []

# end = INITIAL_TRAIN_NCV
# while end + GAP_NCV + TEST_SIZE_NCV <= len(X_jpm):
#     X_tr = X_jpm.iloc[:end]
#     y_tr = y_jpm.iloc[:end]
#     X_te = X_jpm.iloc[end + GAP_NCV : end + GAP_NCV + TEST_SIZE_NCV]
#     y_te = y_jpm.iloc[end + GAP_NCV : end + GAP_NCV + TEST_SIZE_NCV]
#
#     best_name, best_inner, best_fitted = None, -np.inf, None
#     for name, (pipe, grid) in candidates.items():
#         gs = GridSearchCV(clone(pipe), grid, cv=TimeSeriesSplit(n_splits=3),
#                           scoring='roc_auc', n_jobs=-1)
#         gs.fit(X_tr, y_tr)
#         if gs.best_score_ > best_inner:
#             best_name, best_inner, best_fitted = name, gs.best_score_, gs
#
#     auc_outer = roc_auc_score(y_te, best_fitted.predict_proba(X_te)[:, 1])
#     nested_results.append({
#         'fold':       len(nested_results) + 1,
#         'test_start': X_te.index[0].strftime('%Y-%m'),
#         'best_model': best_name,
#         'auc_inner':  best_inner,
#         'auc_outer':  auc_outer,
#         'brecha':     best_inner - auc_outer,
#     })
#     end += STEP_NCV

# df_nested = pd.DataFrame(nested_results)
# display(df_nested.tail(10))

### 3.3 — Visualización e interpretación *(5 pts)*

Genera tres gráficos:
1. Barplot del porcentaje de folds ganados por modelo
2. Boxplot de la brecha `inner − outer` por modelo
3. Serie temporal del `auc_outer` por fold, coloreando según modelo ganador

In [ ]:
# TODO: genera las tres visualizaciones
# fig, axes = plt.subplots(1, 3, figsize=(18, 5))
#
# Paleta para los cinco modelos:
# COLOR_MAP = {
#     'LogisticRegression': '#2196F3',
#     'SVM':                '#9C27B0',
#     'RandomForest':       '#4CAF50',
#     'GradientBoosting':   '#FF9800',
#     'XGBoost':            '#F44336',
# }
#
# Grafico 1 — Barplot porcentaje de folds ganados por modelo
# wins = df_nested['best_model'].value_counts(normalize=True)
# axes[0].bar(wins.index, wins.values,
#             color=[COLOR_MAP[m] for m in wins.index], edgecolor='white')
# axes[0].set_ylabel('Fraccion de folds ganados')
# axes[0].set_title('Dominio por modelo')
#
# Grafico 2 — Boxplot brecha inner-outer por modelo
# import seaborn as sns
# sns.boxplot(data=df_nested, x='best_model', y='brecha',
#             palette=COLOR_MAP, ax=axes[1])
# axes[1].axhline(0.05, ls='--', color='gray', label='Umbral 0.05')
# axes[1].set_title('Brecha inner - outer por modelo')
#
# Grafico 3 — Serie temporal AUC outer coloreado por modelo ganador
# for i, row in df_nested.iterrows():
#     axes[2].scatter(row['test_start'], row['auc_outer'],
#                     color=COLOR_MAP[row['best_model']], s=30, zorder=3)
# axes[2].axhline(0.5, ls='--', color='gray', lw=1)
# axes[2].set_title('AUC outer por fold (color = modelo ganador)')
pass

**Responde (2–3 oraciones por pregunta):**

**a)** ¿Qué modelo domina el mayor porcentaje de folds? ¿Es consistente con su brecha `inner−outer`?

> *Completa aquí*

**b)** Compara el comportamiento de los modelos lineales (LogisticRegression, SVM lineal) contra los de ensemble (RandomForest, GradientBoosting, XGBoost). ¿Cuál familia muestra mayor estabilidad temporal en el AUC outer? ¿A qué lo atribuyes?

> *Completa aquí*

**c)** ¿Identificas algún periodo donde el modelo ganador cambia frecuentemente entre familias (lineal vs ensemble)? ¿A qué evento de mercado podría asociarse ese cambio de régimen?

> *Completa aquí*

**d)** Si un fondo te pidiera elegir **un solo modelo** para desplegar en producción, ¿cuál elegirías basado en los resultados del Nested CV? Justifica con al menos **tres criterios cuantitativos** (ej: AUC outer medio, brecha inner−outer, fracción de folds ganados, std del AUC).

> *Completa aquí*

---
## Parte 4 — Análisis de Sensibilidad del Gap *(20 pts)*

### Contexto

El gap es el parámetro más ignorado en Walk-Forward. Cuando los features usan ventanas rolling largas, su omisión introduce leakage silencioso. Construirás evidencia empírica usando datos de **GLD (2015–2024)** (el oro presenta regímenes de vuelo a la calidad durante crisis que lo hacen interesante).

### 4.1 — Experimento de sensibilidad *(12 pts)*

Para cada `gap ∈ {0, 1, 5, 10, 20, 60}`, ejecuta Walk-Forward Expanding y registra AUC media, std y porcentaje de folds con AUC > 0.5.

In [ ]:
raw_gld = yf.download('GLD', start='2015-01-01', end='2024-12-31',
                      auto_adjust=True, progress=False)
ret_gld = raw_gld['Close'].pct_change().dropna()
ret_gld.name = 'ret'
print(f'GLD: {ret_gld.index[0].date()} — {ret_gld.index[-1].date()} | {len(ret_gld):,} obs')


def build_features_gld(ret, horizonte=1):
    """
    Features para GLD con ventana rolling maxima de 60 dias.
    Incluye: lags 1, 2, 5; vol_10, vol_20, vol_60; rsi_14; label.
    """
    # TODO: implementa los features
    # La ventana mas larga debe ser vol_60 = ret.shift(1).rolling(60).std()
    pass


# feat_gld = build_features_gld(ret_gld)
# X_gld = feat_gld.drop(columns=['label'])
# y_gld = feat_gld['label']
# print(f'GLD: {X_gld.shape[0]} obs | {X_gld.shape[1]} features')

In [ ]:
GAPS_TO_TEST      = [0, 1, 5, 10, 20, 60]
INITIAL_TRAIN_GAP = 504
TEST_SIZE_GAP     = 63
STEP_GAP          = 21

pipe_gap = Pipeline([
    ('imp',    SimpleImputer()),
    ('scaler', StandardScaler()),
    ('model',  LogisticRegression(C=0.1, max_iter=1000)),
])

gap_results = {}

for gap in GAPS_TO_TEST:
    aucs_g = []
    # TODO: ejecuta Walk-Forward Expanding con el gap actual sobre X_gld, y_gld

    gap_results[gap] = {
        'auc_mean': np.mean(aucs_g) if aucs_g else np.nan,
        'auc_std':  np.std(aucs_g)  if aucs_g else np.nan,
        'n_folds':  len(aucs_g),
        'pct_gt05': np.mean([a > 0.5 for a in aucs_g]) if aucs_g else np.nan,
    }
    print(f"Gap={gap:>3} | AUC={gap_results[gap]['auc_mean']:.4f} "
          f"| std={gap_results[gap]['auc_std']:.4f} "
          f"| folds={gap_results[gap]['n_folds']}")

# df_gap = pd.DataFrame(gap_results).T
# df_gap.index.name = 'gap'
# display(df_gap.round(4))

### 4.2 — Visualización e interpretación *(8 pts)*

Genera un gráfico con dos subplots:
1. AUC media ± 1 std vs gap (con bandas de error mediante `errorbar`); marca con línea vertical el gap mínimo teórico
2. Porcentaje de folds con AUC > 0.5 vs gap

In [ ]:
# TODO: genera el gráfico de sensibilidad
# GAP_TEORICO = max(L_max, h) con L_max=60, h=1
# fig, axes = plt.subplots(1, 2, figsize=(12, 4))
# axes[0].errorbar(df_gap.index, df_gap['auc_mean'], yerr=df_gap['auc_std'], ...)
# axes[0].axvline(x=GAP_TEORICO, ls='--', color='red', label=f'Gap teorico={GAP_TEORICO}')
pass

**Responde:**

**a)** Calcula el gap mínimo teórico usando la fórmula de la clase (horizonte = 1, ventana máxima = 60):

$$\text{Gap}_{\min} = \max(L_{\max},\; h) = \max(\;?,\; 1) = ?$$

> *Completa aquí*

**b)** ¿El experimento empírico confirma la predicción teórica? Describe qué observas para gap < gap_mínimo vs gap ≥ gap_mínimo.

> *Completa aquí*

**c)** Un analista propone usar `gap=0` argumentando que "el modelo aprende más con datos adicionales". Refuta este argumento con los resultados de esta sección.

> *Completa aquí*

---
## Parte 5 — Detección de Cambio de Régimen y Modelo Adaptativo *(Bonus — 15 pts)*

### Contexto

Un modelo estable durante 2015–2019 puede degradarse en 2020 (COVID) o 2022 (crisis de tasas). Esta parte explora una estrategia adaptativa: **reentrenar solo cuando se detecta degradación estadística**.

### 5.1 — Detección de degradación por control estadístico *(5 pts)*

Define un fold como "en degradación" si su AUC cae por debajo de `media_rolling − 1.5 × std_rolling`, calculados con los últimos `lookback` folds. Implementa `detect_degradation()` y visualiza los periodos de alerta.

In [ ]:
def detect_degradation(aucs, dates, lookback=10, threshold_sigma=1.5):
    """
    Detecta folds con degradacion estadistica del AUC.

    Para el fold i:
        ventana = aucs[max(0, i-lookback) : i]
        threshold_i = mean(ventana) - threshold_sigma * std(ventana)
        is_degraded = aucs[i] < threshold_i

    Returns DataFrame con columnas: [date, auc, threshold, is_degraded]
    """
    # TODO
    pass


# Usa los resultados de Parte 2 (aucs_expanding, dates_expanding)
# degradation_df = detect_degradation(aucs_expanding, dates_expanding)
# print(f'Folds en degradacion: {degradation_df["is_degraded"].sum()} '
#       f'({degradation_df["is_degraded"].mean():.1%})')

### 5.2 — Estrategia adaptativa vs Walk-Forward estándar *(10 pts)*

Implementa una estrategia que:
1. Usa el modelo del fold anterior por defecto
2. Reentrana **solo cuando `is_degraded` fue True** en el fold anterior
3. Compara AUC medio y número de reentrenamientos vs Walk-Forward estándar

¿La estrategia adaptativa mejora la estabilidad del AUC? ¿Cuál es el costo de cómputo relativo?

In [ ]:
# TODO: implementa la estrategia adaptativa
#
# Pseudocodigo:
#   model_current = None
#   retrain_flag  = True   <- entrena siempre en el primer fold
#   n_retrains    = 0
#   aucs_adaptive = []
#
#   for fold (X_train, y_train, X_test, y_test, is_prev_degraded):
#       if retrain_flag or model_current is None:
#           model_current = clone(pipe_base).fit(X_train, y_train)
#           n_retrains += 1
#       auc_i = roc_auc_score(y_test, model_current.predict_proba(X_test)[:, 1])
#       aucs_adaptive.append(auc_i)
#       retrain_flag = es_degradado(auc_i, umbral)  # decide si entrenar en el proximo fold

pass

# Reporte comparativo
# print(f'Walk-Forward estandar : AUC={np.mean(aucs_expanding):.4f} | retrains={len(aucs_expanding)}')
# print(f'Estrategia adaptativa : AUC={np.mean(aucs_adaptive):.4f}  | retrains={n_retrains}')

---
## Checklist de Entrega

Antes de enviar verifica que:

- [ ] El notebook corre de arriba abajo sin errores (`Kernel → Restart & Run All`)
- [ ] No hay `pass` pendientes en las Partes 1–4 (obligatorias)
- [ ] `build_pipeline_good()` no contiene ninguno de los 3 leakages de `build_pipeline_bad()`
- [ ] El test unitario de `WalkForwardRolling` pasa sin errores
- [ ] El diccionario `candidates` contiene los 5 modelos: LogisticRegression, SVM, RandomForest, GradientBoosting, XGBoost
- [ ] Todas las preguntas textuales tienen respuestas completas (no placeholders)
- [ ] Todos los gráficos tienen título, etiquetas en ejes y leyenda
- [ ] El repositorio tiene al menos 6 commits con conventional commits
- [ ] Los datos se descargan con `yfinance` (no archivos CSV locales en el repo)

---

## Tabla de Puntajes

| Parte | Descripción | Pts |
|---|---|---|
| 1.1 | Identificación de leakages (tabla) | 8 |
| 1.2 | Análisis de leakages | 8 |
| 1.3 | Pipeline corregido con `sklearn.Pipeline` | 9 |
| 2.1 | Features AAPL anti-leakage (≥8) | 5 |
| 2.2 | Clase `WalkForwardRolling` + test unitario | 15 |
| 2.3 | Comparación Expanding vs Rolling | 10 |
| 3.1 | Features JPM cross-asset (≥10) | 8 |
| 3.2 | Nested CV con 5 modelos (LR, SVM, RF, GB, XGB) | 12 |
| 3.3 | Visualizaciones e interpretación (4 preguntas) | 5 |
| 4.1 | Experimento sensibilidad gap | 12 |
| 4.2 | Visualización e interpretación gap | 8 |
| **Total obligatorio** | | **100** |
| Bonus 5.1 | Detección de cambio de régimen | 5 |
| Bonus 5.2 | Modelo adaptativo | 10 |

---

**Archivo a entregar:** URL del repositorio GitHub enviada a `anmedinas@gmail.com`  
**Asunto del mail:** `[HW03] walk_forward_<tu_apellido>`